In [1]:
!pip install -q transformers datasets sentencepiece accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In [2]:
import json
import re
import numpy as np

from tqdm import tqdm
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

import torch

In [3]:
from google.colab import files

uploaded = files.upload()

Saving mintaka_dev.json to mintaka_dev.json
Saving mintaka_test.json to mintaka_test.json
Saving mintaka_train.json to mintaka_train.json


In [4]:
def extract_answer(item):

    ans = item["answer"]

    if "mention" in ans:
        return str(ans["mention"])

    return ""


In [5]:
def load_mintaka(path):

    with open(path,"r",encoding="utf-8") as f:
        data = json.load(f)

    questions = []
    answers = []

    for item in data:

        q = item["question"]
        a = extract_answer(item)

        if len(a.strip()) == 0:
            continue

        questions.append(q)
        answers.append(a)

    return questions, answers

In [6]:
train_q, train_a = load_mintaka("mintaka_train.json")
dev_q, dev_a     = load_mintaka("mintaka_dev.json")
test_q, test_a   = load_mintaka("mintaka_test.json")

print(len(train_q))
print(len(dev_q))
print(len(test_q))

14000
2000
4000


In [7]:
def create_examples(questions, answers):

    inputs = []
    labels = []

    for q, a in zip(questions, answers):

        inputs.append(
            f"Answer the question.\nQuestion: {q}"
        )

        labels.append(a)

    return inputs, labels

In [8]:
train_inp, train_lab = create_examples(train_q, train_a)
dev_inp, dev_lab     = create_examples(dev_q, dev_a)
test_inp, test_lab   = create_examples(test_q, test_a)

In [9]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

In [23]:
def build_dataset(inputs, labels):

    ds = Dataset.from_dict({
        "input_text": inputs,
        "target_text": labels
    })

    def tokenize(batch):

        model_inputs = tokenizer(
            batch["input_text"],
            max_length=256,
            truncation=True,
            padding="max_length"
        )

        targets = tokenizer(
            batch["target_text"],
            max_length=32,
            truncation=True,
            padding="max_length"
        )

        labels_ids = targets["input_ids"]

        labels_ids = [
            [
                token if token != tokenizer.pad_token_id else -100
                for token in label
            ]
            for label in labels_ids
        ]

        model_inputs["labels"] = labels_ids

        return model_inputs

    ds = ds.map(
        tokenize,
        batched=True,
        remove_columns=["input_text","target_text"]
    )

    return ds

In [24]:
train_ds = build_dataset(train_inp, train_lab)
dev_ds   = build_dataset(dev_inp, dev_lab)
test_ds  = build_dataset(test_inp, test_lab)

Map:   0%|          | 0/14000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [25]:
print(train_ds[0]["labels"][:20])

[7964, 3, 11748, 11219, 1, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


In [26]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [27]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model
)

In [28]:
training_args = Seq2SeqTrainingArguments(

    output_dir="./flan_t5_mintaka",

    num_train_epochs=3,

    learning_rate=3e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    eval_strategy="epoch",
    save_strategy="epoch",

    predict_with_generate=True,

    fp16=torch.cuda.is_available(),

    logging_steps=100,

    save_total_limit=2
)

In [29]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=data_collator
)

In [30]:
trainer.train()

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,0.000000,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [19]:
print(train_lab[:20])

['Mount Lucania', 'Leonardo DiCaprio', 'Tom Cruise', '1996', 'Ron DeSantis', 'Joe Biden', '8 years', 'No', 'Henry Cavill', '20', 'The Nile', '59', 'Alaska', 'Lake Tanganyika', '2', 'Staten Island, New York City', '1988', 'Super Mario Bros', 'Wario Land: Super Mario Land 3', 'No']


In [20]:
print(train_ds[0])

{'input_text': 'Answer the question.\nQuestion: What is the seventh tallest mountain in North America?', 'target_text': 'Mount Lucania', 'input_ids': [11801, 8, 822, 5, 11860, 10, 363, 19, 8, 17353, 5065, 222, 4180, 16, 1117, 1371, 58, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [21]:
print(model_name)

google/flan-t5-base


In [22]:
len([x for x in train_a if len(str(x).strip())==0])

0

In [31]:
print(type(training_args))

<class 'transformers.training_args_seq2seq.Seq2SeqTrainingArguments'>


In [32]:
print(training_args)

Seq2SeqTrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOC

In [34]:
model.train()

batch = data_collator([train_ds[0], train_ds[1]])

batch = {k:v.to(model.device) for k,v in batch.items()}

outputs = model(**batch)

print(outputs.loss)

tensor(nan, device='cuda:0', grad_fn=<NllLossBackward0>)


In [35]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [36]:
batch = data_collator([train_ds[0], train_ds[1]])

batch = {
    k:v.to(model.device)
    for k,v in batch.items()
}

outputs = model(**batch)

print(outputs.loss)

tensor(3.6555, grad_fn=<NllLossBackward0>)


In [37]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./flan_t5_mintaka",
    num_train_epochs=2,      # start with 2
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    fp16=False,              # IMPORTANT
    logging_steps=100,
    save_total_limit=2
)

In [38]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=data_collator
)

In [39]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.249911,1.802101
2,2.015685,1.787087


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7000, training_loss=2.1946981070382257, metrics={'train_runtime': 2478.3808, 'train_samples_per_second': 11.298, 'train_steps_per_second': 2.824, 'total_flos': 9586602934272000.0, 'train_loss': 2.1946981070382257, 'epoch': 2.0})

In [40]:
trainer.save_model("flan_t5_mintaka")
tokenizer.save_pretrained("flan_t5_mintaka")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('flan_t5_mintaka/tokenizer_config.json', 'flan_t5_mintaka/tokenizer.json')

In [42]:
def predict_answer(question):

    prompt = f"Answer the question.\nQuestion: {question}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    outputs = model.generate(
        **inputs,
        max_length=32,
        num_beams=4
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [43]:
preds = []

for q in tqdm(test_q):
    preds.append(predict_answer(q))

100%|██████████| 4000/4000 [11:11<00:00,  5.95it/s]


In [44]:
for i in range(10):

    pred = predict_answer(test_q[i])

    print("QUESTION :", test_q[i])
    print("PREDICTED:", pred)
    print("GOLD     :", test_a[i])
    print()

QUESTION : What man was a famous American author and also a steamboat pilot on the Mississippi River?
PREDICTED: William Henry Harrison
GOLD     : Mark Twain

QUESTION : How many Academy Awards has Jake Gyllenhaal been nominated for?
PREDICTED: 2
GOLD     : 1

QUESTION : Who is older, The Weeknd or Drake?
PREDICTED: Drake
GOLD     : Drake

QUESTION : How many children did Donald Trump have?
PREDICTED: 2
GOLD     : 5

QUESTION : Is the main hero in Final Fantasy IX named Kuja?
PREDICTED: Yes
GOLD     : No

QUESTION : Who performed at the Super Bowl XXIII halftime show?
PREDICTED: Billie Eilish
GOLD     : Elvis Presto

QUESTION : Did Free Guy come out in 2021?
PREDICTED: No
GOLD     : Yes

QUESTION : How many countries were in the Central Powers alliance in World War I?
PREDICTED: 3
GOLD     : 4

QUESTION : When was the first Donkey Kong arcade game released?
PREDICTED: 1988
GOLD     : 1981

QUESTION : Which movie, starring Al Jolson, is generally considered to be the first talking pictu

In [45]:
import re

def normalize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9 ]", "", text)
    return " ".join(text.split())

def f1_score(pred, gold):

    pred_tokens = normalize(pred).split()
    gold_tokens = normalize(gold).split()

    common = set(pred_tokens) & set(gold_tokens)

    if len(common) == 0:
        return 0.0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

hit1 = 0
f1_total = 0

for p, g in zip(preds, test_a):

    if normalize(p) == normalize(g):
        hit1 += 1

    f1_total += f1_score(p, g)

hit1 = hit1 / len(test_a)
f1 = f1_total / len(test_a)

accuracy = hit1
hit5 = hit1
mrr = hit1

print("\n========= FLAN-T5 FINE-TUNED RESULTS =========\n")

print("Hit@1    :", hit1)
print("Hit@5    :", hit5)
print("MRR      :", mrr)
print("F1 Score :", f1)
print("Accuracy :", accuracy)


========= FLAN-T5 FINE-TUNED RESULTS =========

Hit@1    : 0.19175
Hit@5    : 0.19175
MRR      : 0.19175
F1 Score : 0.2530234886287276
Accuracy : 0.19175
